### RAG Pipelines: data ingestoin to vector DB pipeline

In [9]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
import os
from pathlib import Path


In [10]:
def process_all_pdfs(pdf_dir):
    all_documents = []
    pdf_dir = Path(pdf_dir)
    pdf_files = list(pdf_dir.glob("*.pdf"))
    print(f"Found {len(pdf_files)} PDF files in {pdf_dir}")
    for pdf_file in pdf_files:
        print(f"Processing {pdf_file}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            # add source information to the metadata
            for document in documents:
                document.metadata["file_name"] = pdf_file.name
                document.metadata["source"] = str(pdf_file)
                document.metadata["file_type"] =  'pdf'
            all_documents.extend(documents)
            print(f"Added {len(documents)} pages from {pdf_file}")
        except Exception as e:
            print(f"Error processing {pdf_file}: {e}")
    print(f"Total documents loaded: {len(all_documents)}")
    return all_documents

all_documents = process_all_pdfs("../data/pdf")
print(f"Found {len(all_documents)} documents")



Found 2 PDF files in ..\data\pdf
Processing ..\data\pdf\MicroServices_Concepts.pdf
Added 18 pages from ..\data\pdf\MicroServices_Concepts.pdf
Processing ..\data\pdf\SiddhantSanchetik_Resume.pdf
Added 2 pages from ..\data\pdf\SiddhantSanchetik_Resume.pdf
Total documents loaded: 20
Found 20 documents


In [11]:
all_documents

[Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\MicroServices_Concepts.pdf', 'total_pages': 18, 'page': 0, 'page_label': '1', 'file_name': 'MicroServices_Concepts.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\MicroServices_Concepts.pdf', 'total_pages': 18, 'page': 1, 'page_label': '2', 'file_name': 'MicroServices_Concepts.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\MicroServices_Concepts.pdf', 'total_pages': 18, 'page': 2, 'page_label': '3', 'file_name': 'MicroServices_Concepts.pdf', 'file_type': 'pdf'}, page_content=''),
 Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'source': '..\\data\\pdf\\MicroServices_Concepts.pdf', 'total_pages': 18, 'page': 3, 'page_label': '4', 'file_name

In [12]:
### Text Splitter
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len, separators=["\n\n", "\n", " ", ""])

    split_docs = text_splitter.split_documents(all_documents)
    print(f"Split {len(split_docs)} documents into {len(split_docs)} chunks")

    # show example of a chunk
    if split_docs:
        print(f"Example of a chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    return split_docs


In [13]:
chunks = split_documents(all_documents)
chunks

Split 9 documents into 9 chunks
Example of a chunk:
Content: SIDDHANT K. SANCHETI
Software Engineer | Full Stack, AI/ML & Distributed Systems
(669) 319-8812 • siddhantsanchetik@gmail.com • linkedin.com/in/siddhant-sancheti • siddhantsancheti.com
Summary
Softwar...
Metadata: {'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-23T23:14:26+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-23T23:14:26+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\SiddhantSanchetik_Resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'file_name': 'SiddhantSanchetik_Resume.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-23T23:14:26+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-23T23:14:26+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\SiddhantSanchetik_Resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'file_name': 'SiddhantSanchetik_Resume.pdf', 'file_type': 'pdf'}, page_content='SIDDHANT K. SANCHETI\nSoftware Engineer | Full Stack, AI/ML & Distributed Systems\n(669) 319-8812 • siddhantsanchetik@gmail.com • linkedin.com/in/siddhant-sancheti • siddhantsancheti.com\nSummary\nSoftware Engineer with 3+ years designing distributed systems, backend services, full-stack products, and production multi-agent AI, backed by evals,\nreliability guardrails, and measurable impact. At AWS, co-designed a multi-agent orchestration system adopted 

In [14]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.utils import embedding_functions
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity


In [15]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)
        self._load_model()

    def _load_model(self):
        try:
            print(f"Loading model {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise e
    
    def generate_embeddings(self, texts):
        if not self.model:
            raise ValueError("Model not loaded. Please load the model first.")
        print(f"Generating embeddings for {len(texts)} texts")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated successfully. Shape: {embeddings.shape}")
        return embeddings
    
## initialize embedding manager
embedding_manager = EmbeddingManager()

# embeddings = embedding_manager.generate_embeddings(chunks)
embedding_manager

# ## initialize chroma client
# client = chromadb.Client()
# client

# ## initialize chroma collection
# collection = client.create_collection("pdf_embeddings")
# collection

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4941.53it/s]


Loading model all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11317.54it/s]


Model loaded successfully. Embedding dimension: 384


In [18]:
###  VectorStore
#### manages document embeddings in a chroma db store
class VectorStore:
        
    def __init__(self, collection_name="pdf_documents", persistent_dir="/data/vector_store"):
        self.collection_name = collection_name
        self.persistent_dir = persistent_dir
        self.client = None
        self.collection = None
        self.initialize_store()


    def initialize_store(self):
        """Initialize the chroma client and collection"""
        try:
            # Create persistent Chroma DB client
            os.makedirs(self.persistent_dir, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persistent_dir)
            print(f"Chroma DB client initialized successfully. Persistent directory: {self.persistent_dir}")

            # Get or create collection
            self.collection = self.client.get_or_create_collection(self.collection_name, metadata={"description": "PDF documents embeddings for RAG"})
            print(f"Vector store initialized successfully. Collection name: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing Chroma DB client: {e}")
            raise e

    def add_documents(self, documents, embeddings):
        """Add documents and embeddings to the vector store"""
        if len(documents) != len(embeddings):
            raise ValueError("The number of documents and embeddings must be the same")

        print(f"Adding {len(documents)} documents to the vector store")

        # Prepare data for chroma db
        ids = []
        metadatas = []
        documents_text  = []
        embeddings_list = []
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            #Generate a unique id for the document
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            documents.append(doc.page_content)


            documents_text.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
        if self.collection is None:
            raise RuntimeError("Vector store collection is not initialized")
        try:
            self.collection.add(
                ids=ids,
                documents=documents_text,
                metadatas=metadatas,
                embeddings=embeddings_list
            )
            print(f"Documents added to the vector store successfully. Number of documents: {len(documents)}")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents to the vector store: {e}")
            raise e

vector_store = VectorStore()
vector_store

Chroma DB client initialized successfully. Persistent directory: /data/vector_store
Vector store initialized successfully. Collection name: pdf_documents
Existing documents in collection: 0


In [19]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.24', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-10-23T23:14:26+00:00', 'author': '', 'keywords': '', 'moddate': '2026-10-23T23:14:26+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.24 (TeX Live 2022) kpathsea version 6.3.4', 'subject': '', 'title': '', 'trapped': '/False', 'source': '..\\data\\pdf\\SiddhantSanchetik_Resume.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'file_name': 'SiddhantSanchetik_Resume.pdf', 'file_type': 'pdf'}, page_content='SIDDHANT K. SANCHETI\nSoftware Engineer | Full Stack, AI/ML & Distributed Systems\n(669) 319-8812 • siddhantsanchetik@gmail.com • linkedin.com/in/siddhant-sancheti • siddhantsancheti.com\nSummary\nSoftware Engineer with 3+ years designing distributed systems, backend services, full-stack products, and production multi-agent AI, backed by evals,\nreliability guardrails, and measurable impact. At AWS, co-designed a multi-agent orchestration system adopted 

In [ ]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]
### Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)


vector_store.add_documents(chunks, embeddings)

Generating embeddings for 9 texts


Batches: 100%|██████████| 1/1 [00:00<00:00,  5.21it/s]

Embeddings generated successfully. Shape: (9, 384)
Adding 9 documents to the vector store


Documents added to the vector store successfully. Number of documents: 18
Total documents in collection: 9
